# 7B инференс по готовому адаптеру (без обучения)

Восстанавливает `query_results_{baseline,lora,zeroshot}.json` из уже обученного адаптера —
SFT не запускается. Генерация greedy (детерминирована), поэтому числа совпадут с тем,
что дал бы обучающий прогон.

**Перед запуском:** GPU On + Internet On. Подключи два датасета:
1. данные (`bird_large.json` + профили) — тот же sql_only, что для обучения;
2. `adapter_synth1096_7b.zip` — как отдельный Kaggle Dataset.
Поправь `DATA_DIR` и `ADAPTER_ZIP` ниже.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate

In [ ]:
import json, re, torch, zipfile
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

# ПОПРАВЬ пути под свои датасеты на Kaggle:
DATA_DIR    = Path("/kaggle/input/datasets/vorange/db2model-lora")   # bird_large.json + *_profile.json
ADAPTER_ZIP = Path("/kaggle/input/adapter-synth1096-7b/adapter_synth1096_7b.zip")  # твой адаптер

OUT_DIR = Path("/kaggle/working")
ADAPTER_DIR = OUT_DIR / "adapter"
with zipfile.ZipFile(ADAPTER_ZIP) as z:
    z.extractall(ADAPTER_DIR)
print("адаптер распакован:", sorted(p.name for p in ADAPTER_DIR.iterdir()))

bird = json.loads((DATA_DIR / "bird_large.json").read_text(encoding="utf-8"))
DBS = ["financial", "toxicology", "codebase_community"]
profiles = {db: json.loads((DATA_DIR / f"{db}_profile.json").read_text(encoding="utf-8"))
            for db in DBS}
print("GPU:", torch.cuda.get_device_name(0), "| базы:", DBS)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
load_kwargs = dict(quantization_config=bnb, device_map={"": 0})
try:
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=COMPUTE_DTYPE, **load_kwargs)
except TypeError:
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE, **load_kwargs)

# Прикручиваем LoRA-адаптер к базовой модели — это инференс, обучения нет.
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
model.config.use_cache = True
print("память на GPU:", round(torch.cuda.memory_allocated() / 1e9, 2), "ГБ")

In [ ]:
def schema_text(db: str) -> str:
    lines = []
    for table, info in profiles[db]["tables"].items():
        cols = ", ".join(f"{c['name']} {c['type']}" for c in info["columns"])
        lines.append(f"{table}({cols})")
    return "\n".join(lines)

def build_prompt(db: str, question: str, with_schema: bool) -> str:
    system = f"You are a PostgreSQL expert for the database `{db}`. Return only SQL."
    if with_schema:
        system += f"\n\nSchema:\n{schema_text(db)}"
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True)

def generate(gen_model, prompt: str, max_new_tokens: int = 200):
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, inputs["input_ids"].shape[1]

FENCED = re.compile(r"```(?:sql)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)
STATEMENT = re.compile(r"\b(WITH|SELECT)\b", re.IGNORECASE)

def clean_sql(text: str) -> str:
    text = text.strip()
    fenced = FENCED.search(text)
    if fenced:
        text = fenced.group(1).strip()
    start = STATEMENT.search(text)
    if start:
        text = text[start.start():]
    return text.strip().rstrip(";").strip()

In [ ]:
import contextlib

questions = [q for q in bird if q["db_id"] in DBS]
print("вопросов:", len(questions))

def run_arm(name: str, with_schema: bool, use_adapter: bool) -> dict:
    predictions, prompt_tokens = {}, []
    ctx = contextlib.nullcontext() if use_adapter else model.disable_adapter()
    with ctx:
        for i, q in enumerate(questions, 1):
            question = f"question: {q['question']}, evidence (may be empty): {q['evidence']}"
            prompt = build_prompt(q["db_id"], question, with_schema)
            text, n_tok = generate(model, prompt)
            predictions[str(q["question_id"])] = clean_sql(text)
            prompt_tokens.append(n_tok)
            if i % 20 == 0:
                print(f"  {name}: {i}/{len(questions)}")
    out = OUT_DIR / f"query_results_{name}.json"
    out.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
    avg = sum(prompt_tokens) / len(prompt_tokens)
    print(f"{name}: {out} | prompt-токенов в среднем {avg:.0f}")
    return {"file": str(out), "avg_prompt_tokens": avg}

stats = {
    "baseline": run_arm("baseline", with_schema=True,  use_adapter=False),
    "lora":     run_arm("lora",     with_schema=False, use_adapter=True),
    "zeroshot": run_arm("zeroshot", with_schema=False, use_adapter=False),
}
(OUT_DIR / "token_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
print("\nготово — забирай query_results_{baseline,lora,zeroshot}.json")